In [1]:
import os, sys
# --- Posez les variables AVANT d'importer torch ---
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"
os.environ["AMD_SERIALIZE_KERNEL"] = "3"
os.environ["PYTORCH_SDPA_ENABLE_HEURISTIC"] = "0"
os.environ["PYTORCH_SDPA_ALLOW_MATH"] = "1"
os.environ["PYTORCH_SDPA_ENABLE_FLASH"] = "0"
os.environ["PYTORCH_SDPA_ENABLE_MEM_EFFICIENT"] = "0"
os.environ["HIP_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_HIP_ALLOC_CONF"]="expandable_segments:True"

import torch
print("torch.version.hip:", torch.version.hip)
print("env HSA_OVERRIDE_GFX_VERSION:", os.environ.get("HSA_OVERRIDE_GFX_VERSION"))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device 0:", torch.cuda.get_device_name(0))

torch.version.hip: 6.2.41133-dd7f95766
env HSA_OVERRIDE_GFX_VERSION: 10.3.0
CUDA available: True
Device 0: AMD Radeon 680M


In [2]:
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer
import nltk
nltk.download('punkt_tab')

/home/mauceric/lora_local/llenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/mauceric/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
max_tokens = 4000
min_tokens = 60
sim_threshold = 0.55
batch_size = 32


In [4]:
# Téléchargement du tokenizer de phrases NLTK
nltk.download("punkt", quiet=True)

# Initialisation du modèle d'embedding
model = SentenceTransformer(model_name)

# On peut aussi préciser explicitement le device si besoin :
# model = SentenceTransformer(model_name, device="cuda")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1695.36it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/mauceric/lora_local/llenv/lib/python3.12/site-packages/torch/nn/modules/module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at ../c10/hip/HIPAllocatorConfig.h:29.)
  return t.to(


In [5]:
def sentence_split(text: str) -> List[str]:
    """
    Découpe un texte en phrases non vides.
    """
    sentences = nltk.sent_tokenize(text)
    return [s.strip() for s in sentences if s.strip()]


In [6]:
def encode_sentences(sentences: List[str]) -> np.ndarray:
    """
    Calcule les embeddings normalisés des phrases
    avec le modèle SentenceTransformer global.
    """
    embeddings = model.encode(
        sentences,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return embeddings


In [7]:
def count_tokens_approx(text: str) -> int:
    """
    Approximation simple du nombre de tokens par le nombre de mots.
    À remplacer par un vrai tokenizer Phi-4 si nécessaire.
    """
    return len(text.split())


In [8]:
def chunk(text: str) -> List[str]:
    """
    Retourne une liste de passages sémantiquement cohérents
    en respectant les contraintes max_tokens / min_tokens.
    """
    sentences = sentence_split(text)
    if not sentences:
        return []

    embeddings = encode_sentences(sentences)

    chunks: List[str] = []
    current_sents: List[str] = []
    current_embs: List[np.ndarray] = []
    current_tokens = 0

    for sent, emb in zip(sentences, embeddings):
        sent_tokens = count_tokens_approx(sent)

        # Initialisation d'un nouveau chunk si nécessaire
        if not current_sents:
            current_sents.append(sent)
            current_embs.append(emb)
            current_tokens = sent_tokens
            continue

        # Similarité cosinus avec le chunk courant
        chunk_emb = np.mean(np.stack(current_embs, axis=0), axis=0)
        sim = float(np.dot(chunk_emb, emb))  # embeddings normalisés

        # Critères de coupure
        size_limit_reached = (current_tokens + sent_tokens) > max_tokens
        semantic_break = (sim < sim_threshold) and (current_tokens >= min_tokens)

        if size_limit_reached or semantic_break:
            # On ferme le chunk courant
            chunks.append(" ".join(current_sents))
            # On démarre un nouveau chunk
            current_sents = [sent]
            current_embs = [emb]
            current_tokens = sent_tokens
        else:
            # On continue le chunk courant
            current_sents.append(sent)
            current_embs.append(emb)
            current_tokens += sent_tokens

    # Ajout du dernier chunk
    if current_sents:
        chunks.append(" ".join(current_sents))

    return chunks


In [9]:
textes = [
    """
Quand je considère ces têtes 
entassées en ces charniers, 
tous furent maîtres des requêtes, 
au moins de la chambre aux deniers, 
où tous furent porte-paniers, 
car d'évêques ou de lanterniers, 
je ne connais rien à y redire. 
Et ycelles qui s'inclinaient une contre autre en leur vie,
desquelles les unes régnaient des autres craintes et servies,
là les voit toutes assouvies, ensemble en un tas pêle-mêle, 
seigneuries leur sont ravies, clerc ni maître ne s'y appelle. 
Or ils sont morts, Dieu ait leurs âmes, 
quant est des corps ils sont pourris, plaise au doux Jésus les absoudre
""",
    """
    … Une belle journée de mai, un beau soleil, un ciel pur… Quand les canots étrangers arrivèrent, les bourreaux, sur les quais, mettaient la dernière main à leur œuvre : six pendus exécutaient en présence de la foule l’horrible contorsion finale… Les fenêtres, les toits étaient encombrés de spectateurs ; sur un balcon voisin, les autorités turques souriaient à ce spectacle familier.

Le gouvernement du sultan avait fait peu de frais pour l’appareil du supplice ; les potences étaient si basses que les pieds nus des condamnés touchaient la terre. Leurs ongles crispés grinçaient sur le sable.



II
L’exécution terminée, les soldats se retirèrent et les morts restèrent jusqu’à la tombée du jour exposés aux yeux du peuple. Les six cadavres, debout sur leurs pieds, firent, jusqu’au soir, la hideuse grimace de la mort au beau soleil de Turquie, au milieu de promeneurs indifférents et de groupes silencieux de jeunes femmes.



III
Les gouvernements de France et d’Allemagne avaient exigé ces exécutions d’ensemble, comme réparation de ce massacre des consuls qui fit du bruit en Europe au début de la crise orientale.

Toutes les nations européennes avaient envoyé sur rade de Salonique d’imposants cuirassés. L’Angleterre s’y était une des premières fait représenter, et c’est ainsi que j’y étais venu moi-même, sur l’une des corvettes de Sa Majesté.



IV
Un beau jour de printemps, un des premiers où il nous fut permis de circuler dans Salonique de Macédoine, peu après les massacres, trois jours après les pendaisons, vers quatre heures de l’après-midi, il arriva que je m’arrêtai devant la porte fermée d’une vieille mosquée, pour regarder se battre deux cigognes.

La scène se passait dans une rue du vieux quartier musulman. Des maisons caduques bordaient de petits chemins tortueux, à moitié recouverts par les saillies des shaknisirs (sorte d’observatoires mystérieux, de grands balcons fermés et grillés, d’où les passants sont reluqués par des petits trous invisibles). Des avoines poussaient entre les pavés de galets noirs, et des branches de fraîche verdure couraient sur les toits ; le ciel, entrevu par échappées, était pur et bleu ; on respirait partout l’air tiède et la bonne odeur de mai.

La population de Salonique conservait encore envers nous une attitude contrainte et hostile ; aussi l’autorité nous obligeait-elle à traîner par les rues un sabre et tout un appareil de guerre. De loin en loin, quelques personnages à turban passaient en longeant les murs, et aucune tête de femme ne se montrait derrière les grillages discrets des haremlikes ; on eût dit une ville morte.

Je me croyais si parfaitement seul, que j’éprouvai une étrange impression en apercevant près de moi, derrière d’épais barreaux de fer, le haut d’une tête humaine, deux grands yeux verts fixés sur les miens.

Les sourcils étaient bruns, légèrement froncés, rapprochés jusqu’à se rejoindre ; l’expression de ce regard était un mélange d’énergie et de naïveté ; on eût dit un regard d’enfant, tant il avait de fraîcheur et de jeunesse.

La jeune femme qui avait ces yeux se leva, et montra jusqu’à la ceinture sa taille enveloppée d’un camail à la turque (féredjé) aux plis longs et rigides. Le camail était de soie verte, orné de broderies d’argent. Un voile blanc enveloppait soigneusement la tête, n’en laissant paraître que le front et les grands yeux. Les prunelles étaient bien vertes, de cette teinte vert de mer d’autrefois chantée par les poètes d’Orient.

Cette jeune femme était Aziyadé.



V
Aziyadé me regardait fixement. Devant un Turc, elle se fût cachée ; mais un giaour n’est pas un homme ; tout au plus est-ce un objet de curiosité qu’on peut contempler à loisir. Elle paraissait surprise qu’un de ces étrangers, qui étaient venus menacer son pays sur de si terribles machines de fer, pût être un très jeune homme dont l’aspect ne lui causait ni répulsion ni frayeur.



VI
Tous les canots des escadres étaient partis quand je revins sur le quai ; les yeux verts m’avaient légèrement captivé, bien que le visage exquis caché par le voile blanc me fût encore inconnu ; j’étais repassé trois fois devant la mosquée aux cigognes, et l’heure s’en était allée sans que j’en eusse conscience.

Les impossibilités étaient entassées comme à plaisir entre cette jeune femme et moi ; impossibilité d’échanger avec elle une pensée, de lui parler ni de lui écrire ; défense de quitter le bord après six heures du soir, et autrement qu’en armes ; départ probable avant huit jours pour ne jamais revenir, et, par dessus tout, les farouches surveillances des harems.

Je regardai s’éloigner les derniers canots anglais, le soleil près de disparaître, et je m’assis irrésolu sous la tente d’un café turc.



VII
Un attroupement fut aussitôt formé autour de moi ; c’était une bande de ces hommes qui vivent à la belle étoile sur les quais de Salonique, bateliers ou portefaix, qui désiraient savoir pourquoi j’étais resté à terre et attendaient là, dans l’espoir que peut-être j’aurais besoin de leurs services.

Dans ce groupe de Macédoniens, je remarquai un homme qui avait une drôle de barbe, séparée en petites boucles comme les plus antiques statues de ce pays ; il était assis devant moi par terre et m’examinait avec beaucoup de curiosité ; mon costume et surtout mes bottines paraissaient l’intéresser vivement. Il s’étirait avec des airs câlins, des mines de gros chat angora, et bâillait en montrant deux rangées de dents toutes petites, aussi brillantes que des perles.

Il avait d’ailleurs une très belle tête, une grande douceur dans les yeux qui resplendissaient d’honnêteté et d’intelligence. Il était tout dépenaillé, pieds nus, jambes nues, la chemise en lambeaux, mais propre comme une chatte.

Ce personnage était Samuel.



VIII
Ces deux êtres rencontrés le même jour devaient bientôt remplir un rôle dans mon existence et jouer, pendant trois mois, leur vie pour moi ; on m’eût beaucoup étonné en me le disant. Tous deux devaient abandonner ensuite leur pays pour me suivre, et nous étions destinés à passer l’hiver ensemble, sous le même toit, à Stamboul.



IX
Samuel s’enhardit jusqu’à me dire les trois mots qu’il savait d’anglais :

— Do you want to go on board ? (Avez-vous besoin d’aller à bord ?)

Et il continua en sabir :

— Te portarem col la mia barca. (Je t’y porterai avec ma barque.)

Samuel entendait le sabir ; je songeai tout de suite au parti qu’on pouvait tirer d’un garçon intelligent et déterminé, parlant une langue connue, pour cette entreprise insensée qui flottait déjà devant moi à l’état de vague ébauche.

L’or était un moyen de m’attacher ce va-nu-pieds, mais j’en avais peu. Samuel, d’ailleurs, devait être honnête, et un garçon qui l’est ne consent point pour de l’or à servir d’intermédiaire entre un jeune homme et une jeune femme.



X

À WILLIAM BROWN, LIEUTENANT AU 3e D’INFANTERIE DE LIGNE, À LONDRES
Salonique, 2 juin.
… Ce n’était d’abord qu’une ivresse de l’imagination et des sens ; quelque chose de plus est venu ensuite, de l’amour ou peu s’en faut ; j’en suis surpris et charmé.

Si vous aviez pu suivre aujourd’hui votre ami Loti dans les rues d’un vieux quartier solitaire, vous l’auriez vu monter dans une maison d’aspect fantastique. La porte se referme sur lui avec mystère. C’est la case choisie pour ces changements de décors qui lui sont familiers. (Autrefois, vous vous en souvenez, c’était pour Isabelle B…, l’étoile : la scène se passait dans un fiacre, ou Hay-Market street, chez la maîtresse du grand Martyn ; vieille histoire que ces changements de décors, et c’est à peine si le costume oriental leur prête encore quelque peu d’attrait et de nouveauté.)

Début de mélodrame. Premier tableau : Un vieil appartement obscur. Aspect assez misérable, mais beaucoup de couleur orientale. Des narguilhés traînent à terre avec des armes.

Votre ami Loti est planté au milieu et trois vieilles juives s’empressent autour de lui sans mot dire. Elles ont des costumes pittoresques et des nez crochus, de longues vestes ornées de paillettes, des sequins enfilés pour colliers, et, pour coiffure, des catogans de soie verte. Elles se dépêchent de lui enlever ses vêtements d’officier et se mettent à l’habiller à la turque, en s’agenouillant pour commencer par les guêtres dorées et les jarretières. Loti conserve l’air sombre et préoccupé qui convient au héros d’un drame lyrique.

Les trois vieilles mettent dans sa ceinture plusieurs poignards dont les manches d’argent sont incrustés de corail, et les lames damasquinées d’or ; elles lui passent une veste dorée à manches flottantes, et le coiffent d’un tarbouch. Après cela, elles expriment, par des gestes, que Loti est très beau ainsi, et vont chercher un grand miroir.

Loti trouve qu’il n’est pas mal en effet, et sourit tristement à cette toilette qui pourrait lui être fatale ; et puis il disparaît par une porte de derrière et traverse toute une ville saugrenue, des bazars d’Orient et des mosquées ; il passe inaperçu dans des foules bariolées, vêtues de ces couleurs éclatantes qu’on affectionne en Turquie ; quelques femmes voilées de blanc se disent seulement sur son passage : « Voici un Albanais qui est bien mis, et ses armes sont belles. »

Plus loin, mon cher William, il serait imprudent de suivre votre ami Loti ; au bout de cette course, il y a l’amour d’une femme turque, laquelle est la femme d’un Turc, — entreprise insensée en tout temps, et qui n’a plus de nom dans les circonstances du jour. — Auprès d’elle, Loti va passer une heure de complète ivresse, au risque de sa tête, de la tête de plusieurs autres, et de toutes sortes de complications diplomatiques.

Vous direz qu’il faut, pour en arriver là, un terrible fond d’égoïsme ; je ne dis pas le contraire ; mais j’en suis venu à penser que tout ce qui me plaît est bon à faire et qu’il faut toujours épicer de son mieux le repas si fade de la vie.

Vous ne vous plaindrez pas de moi, mon cher William : je vous ai écrit longuement. Je ne crois nullement à votre affection, pas plus qu’à celle de personne ; mais vous êtes, parmi les gens que j’ai rencontrés deçà et delà dans le monde, un de ceux avec lesquels je puis trouver du plaisir à vivre et à échanger mes impressions. S’il y a dans ma lettre quelque peu d’épanchement, il ne faut pas m’en vouloir : j’avais bu du vin de Chypre.

À présent c’est passé ; je suis monté sur le pont respirer l’air vif du soir, et Salonique faisait piètre mine ; ses minarets avaient l’air d’un tas de vieilles bougies, posées sur une ville sale et noire où fleurissent les vices de Sodome. Quand l’air humide me saisit comme une douche glacée, et que la nature prend ses airs ternes et piteux, je retombe sur moi-même ; je ne retrouve plus au-dedans de moi que le vide écœurant et l’immense ennui de vivre.

Je pense aller bientôt à Jérusalem, où je tâcherai de ressaisir quelques bribes de foi. Pour l’instant, mes croyances religieuses et philosophiques, mes principes de morale, mes théories sociales, etc., sont représentés par cette grande personnalité : le gendarme.

Je vous reviendrai sans doute en automne dans le Yorkshire. En attendant, je vous serre les mains et je suis votre dévoué.

LOTI.


XI
Ce fut une des époques troublées de mon existence que ces derniers jours de mai 1876.

Longtemps j’étais resté anéanti, le cœur vide, inerte, à force d’avoir souffert ; mais cet état transitoire avait passé, et la force de la jeunesse amenait le réveil. Je m’éveillais seul dans la vie ; mes dernières croyances s’en étaient allées, et aucun frein ne me retenait plus.

Quelque chose comme de l’amour naissait sur ces ruines, et l’Orient jetait son grand charme sur ce réveil de moi-même, qui se traduisait par le trouble des sens.



XII
Elle était venue habiter avec les trois autres femmes de son maître un yali de campagne, dans un bois, sur le chemin de Monastir ; là, on la surveillait moins.

Le jour je descendais en armes. Par grosse mer, toujours, un canot me jetait sur les quais, au milieu de la foule des bateliers et des pêcheurs ; et Samuel, placé comme par hasard sur mon passage, recevait par signes mes ordres pour la nuit.

J’ai passé bien des journées à errer sur ce chemin de Monastir. C’était une campagne nue et triste, où l’œil s’étendait à perte de vue sur des cimetières antiques ; des tombes de marbre en ruine, dont le lichen rongeait les inscriptions mystérieuses ; des champs plantés de menhirs de granit ; des sépultures grecques, byzantines, musulmanes, couvraient ce vieux sol de Macédoine où les grands peuples du passé ont laissé leur poussière. De loin en loin, la silhouette aiguë d’un cyprès, ou un platane immense, abritant des bergers albanais et des chèvres ; sur la terre aride, de larges fleurs lilas pâle, répandant une douce odeur de chèvrefeuille, sous un soleil déjà brûlant. Les moindres détails de ce pays sont restés dans ma mémoire.

La nuit, c’était un calme tiède, inaltérable, un silence mêlé de bruits de cigales, un air pur rempli de parfums d’été ; la mer immobile, le ciel aussi brillant qu’autrefois dans mes nuits des tropiques.

Elle ne m’appartenait pas encore ; mais il n’y avait plus entre nous que des barrières matérielles, la présence de son maître, et le grillage de fer de ses fenêtres.

Je passais ces nuits à l’attendre, à attendre ce moment, très court quelquefois, où je pouvais toucher ses bras à travers les terribles barreaux, et embrasser dans l’obscurité ses mains blanches, ornées de bagues d’Orient.

Et puis, à certaine heure du matin, avant le jour, je pouvais, avec mille dangers, rejoindre ma corvette par un moyen convenu avec les officiers de garde.



XIII
Mes soirées se passaient en compagnie de Samuel. J’ai vu d’étranges choses avec lui, dans les tavernes des bateliers ; j’ai fait des études de mœurs que peu de gens ont pu faire, dans les cours des miracles et les tapis francs des juifs de la Turquie. Le costume que je promenais dans ces bouges était celui des matelots turcs, le moins compromettant pour traverser de nuit la rade de Salonique. Samuel contrastait singulièrement avec de pareils milieux ; sa belle et douce figure rayonnait sur ces sombres repoussoirs. Peu à peu je m’attachais à lui, et son refus de me servir auprès d’Aziyadé me faisait l’estimer davantage.

Mais j’ai vu d’étranges choses la nuit avec ce vagabond, une prostitution étrange, dans les caves où se consomment jusqu’à complète ivresse le mastic et le raki…



XIV
Une nuit tiède de juin, étendus tous deux à terre dans la campagne, nous attendions deux heures du matin, — l’heure convenue. — Je me souviens de cette belle nuit étoilée, où l’on n’entendait que le faible bruit de la mer calme. Les cyprès dessinaient sur la montagne des larmes noires, les platanes des masses obscures ; de loin en loin, de vieilles bornes séculaires marquaient la place oubliée de quelque derviche d’autrefois ; l’herbe sèche, la mousse et le lichen avaient bonne odeur ; c’était un bonheur d’être en pleine campagne une pareille nuit, et il faisait bon vivre.

Mais Samuel paraissait subir cette corvée nocturne avec une détestable humeur, et ne me répondait même plus.

Alors je lui pris la main pour la première fois, en signe d’amitié, et lui fis en espagnol à peu près ce discours :

— Mon bon Samuel, vous dormez chaque nuit sur la terre dure ou sur des planches ; l’herbe qui est ici est meilleure et sent bon comme le serpolet. Dormez, et vous serez de plus belle humeur après. N’êtes-vous pas content de moi ? et qu’ai-je pu vous faire ?

Sa main tremblait dans la mienne et la serrait plus qu’il n’eût été nécessaire.

— Che volete, dit-il d’une voix sombre et troublée, che volete mî ? (Que voulez-vous de moi ?)…

Quelque chose d’inouï et de ténébreux avait un moment passé dans la tête du pauvre Samuel ; — dans le vieil Orient tout est possible ! — et puis il s’était couvert la figure de ses bras, et restait là, terrifié de lui-même, immobile et tremblant…

Mais, depuis cet instant étrange, il est à mon service corps et âme ; il joue chaque soir sa liberté et sa vie en entrant dans la maison qu’Aziyadé habite ; il traverse, dans l’obscurité, pour aller la chercher, ce cimetière rempli pour lui de visions et de terreurs mortelles ; il rame jusqu’au matin dans sa barque pour veiller sur la nôtre, ou bien m’attend toute la nuit, couché pêle-mêle avec cinquante vagabonds, sur la cinquième dalle de pierre du quai de Salonique. Sa personnalité est comme absorbée dans la mienne, et je le trouve partout dans mon ombre, quels que soient le lieu et le costume, que j’aie choisis, prêt à défendre ma vie au risque de la sienne.



XV

LOTI À PLUMKETT, LIEUTENANT DE MARINE


Salonique, mai 1876.
Mon cher Plumkett,
Vous pouvez me raconter, sans m’ennuyer jamais, toutes les choses tristes ou saugrenues, ou même gaies, qui vous passeront par la tête ; comme vous êtes classé pour moi en dehors du « vil troupeau », je lirai toujours avec plaisir ce que vous m’écrirez.

Votre lettre m’a été remise sur la fin d’un dîner au vin d’Espagne, et je me souviens qu’elle m’a un peu, à première vue, abasourdi par son ensemble original. Vous êtes en effet « un drôle de type » ; mais cela, je le savais déjà. Vous êtes aussi un garçon d’esprit, ce qui était connu. Mais ce n’est point là seulement ce que j’ai démêlé dans votre longue lettre, je vous l’assure.

J’ai vu que vous avez dû beaucoup souffrir, et c’est là un point de commun entre nous deux. Moi aussi, il y a dix longues années que j’ai été lancé dans la vie, à Londres, livré à moi-même à seize ans ; j’ai goûté un peu toutes les jouissances ; mais je ne crois pas non plus qu’aucun genre de douleur m’ait été épargné. Je me trouve fort vieux, malgré mon extrême jeunesse physique, que j’entretiens par l’escrime et l’acrobatie.

Les confidences d’ailleurs ne servent à rien ; il suffit que vous ayez souffert pour qu’il y ait sympathie entre nous.

Je vois aussi que j’ai été assez heureux pour vous inspirer quelque affection ; je vous en remercie. Nous aurons, si vous voulez bien, ce que vous appelez une amitié intellectuelle, et nos relations nous aideront à passer le temps maussade de la vie.

À la quatrième page de votre papier, votre main courait un peu vite sans doute, quand vous avez écrit : « une affection et un dévouement illimités ». Si vous avez pensé cela, vous voyez bien, mon cher ami, qu’il y a encore chez vous de la jeunesse et de la fraîcheur, et que tout n’est pas perdu. Ces belles amitiés-là, à la vie, à la mort, personne plus que moi n’en a éprouvé tout le charme ; mais, voyez-vous, on les a à dix-huit ans ; à vingt-cinq, elles sont finies, et on n’a plus de dévouement que pour soi-même. C’est désolant, ce que je vous dis là, mais c’est terriblement vrai.



XVI
Salonique, juin 1876.
C’était un bonheur de faire à Salonique ces corvées matinales qui vous mettaient à terre avant le lever du soleil. L’air était si léger, la fraîcheur si délicieuse, qu’on n’avait aucune peine à vivre ; on était comme pénétré de bien-être. Quelques Turcs commençaient à circuler, vêtus de robes rouges, vertes ou orange, sous les rues voûtées des bazars, à peine éclairées encore d’une demi-lueur transparente.

L’ingénieur Thompson jouait auprès de moi le rôle du confident d’opéra-comique, et nous avons bien couru ensemble par les vieilles rues de cette ville, aux heures les plus prohibées et dans les tenues les moins réglementaires.

Le soir, c’était pour les yeux un enchantement d’un autre genre : tout était rose ou doré. L’Olympe avait des teintes de braise ou de métal en fusion, et se réfléchissait dans une mer unie comme une glace. Aucune vapeur dans l’air : il semblait qu’il n’y avait plus d’atmosphère et que les montagnes se découpaient dans le vide, tant leurs arêtes les plus lointaines étaient nettes et décidées.

Nous étions souvent assis le soir sur les quais où se portait la foule, devant cette baie tranquille. Les orgues de Barbarie d’Orient y jouaient leurs airs bizarres, accompagnés de clochettes et de chapeaux chinois ; les cafedjis encombraient la voie publique de leurs petites tables toujours garnies, et ne suffisaient plus à servir les narguilhés, les skiros, le lokoum et le raki.

Samuel était heureux et fier quand nous l’invitions à notre table. Il rôdait alentour, pour me transmettre par signes convenus quelque rendez-vous d’Aziyadé, et je tremblais d’impatience en songeant à la nuit qui allait venir.



XVII
Salonique, juillet 1876.
Aziyadé avait dit à Samuel qu’il resterait cette nuit-là auprès de nous. Je la regardais faire avec étonnement : elle m’avait prié de m’asseoir entre elle et lui, et commençait à lui parler en langue turque.

C’était un entretien qu’elle voulait, le premier entre nous deux, et Samuel devait servir d’interprète ; depuis un mois, liés par l’ivresse des sens, sans avoir pu échanger même une pensée, nous étions restés jusqu’à cette nuit étrangers l’un à l’autre et inconnus.

— Où es-tu né ? Où as-tu vécu ? Quel âge as-tu ? As-tu une mère ? Crois-tu en Dieu ? Es-tu allé dans le pays des hommes noirs ? As-tu eu beaucoup de maîtresses ? Es-tu un seigneur dans ton pays ?

Elle, elle était une petite fille circassienne venue à Constantinople avec une autre petite de son âge ; un marchand l’avait vendue à un vieux Turc qui l’avait élevée pour la donner à son fils ; le fils était mort, le vieux Turc aussi ; elle, qui avait seize ans, était extrêmement belle ; alors, elle avait été prise par cet homme, qui l’avait remarquée à Stamboul et ramenée dans sa maison de Salonique.

— Elle dit, traduisait Samuel, que son Dieu n’est pas le même que le tien, et qu’elle n’est pas bien sûre, d’après le Koran, que les femmes aient une âme comme les hommes ; elle pense que, quand tu seras parti, vous ne vous verrez jamais, même après que vous serez morts, et c’est pour cela qu’elle pleure. Maintenant, dit Samuel en riant, elle demande si tu veux te jeter dans la mer avec elle tout de suite ; et vous vous laisserez couler au fond en vous tenant serrés tous les deux… Et moi, ensuite, je ramènerai la barque, et je dirai que je ne vous ai pas vus.

— Moi, dis-je, je le veux bien, pourvu qu’elle ne pleure plus ; partons tout de suite, ce sera fini après.

Aziyadé comprit, elle passa ses bras en tremblant autour de mon cou ; et nous nous penchâmes tous deux sur l’eau.

— Ne faites pas cela, cria Samuel, qui eut peur, en nous retenant tous deux avec une poigne de fer. Vilain baiser que vous vous donneriez là. En se noyant, on se mord et on fait une horrible grimace.

Cela était dit en sabir avec une crudité sauvage que le français ne peut pas traduire.

. . . . . . . . . . . . . . . . . . . . . . . . .

Il était l’heure pour Aziyadé de repartir, et, l’instant d’après, elle nous quitta.



XVIII

PLUMKETT À LOTI
Londres, juin 1876.
Mon cher Loti,
J’ai une vague souvenance de vous avoir envoyé le mois dernier une lettre sans queue ni tête, ni rime ni raison. Une de ces lettres que le primesaut vous dicte, où l’imagination galope, suivie par la plume, qui, elle, ne fait que trotter, et encore en butant souvent comme une vieille rossinante de louage.

Ces lettres-là, on ne les a jamais relues avant de les fermer car alors on ne les aurait point envoyées. Des digressions plus ou moins pédantesques dont il est inutile de chercher l’à-propos, suivies d’âneries indignes du Tintamarre. Ensuite, pour le bouquet, un auto-panégyrique d’individu incompris qui cherche à se faire plaindre, pour récolter des compliments que vous êtes assez bon pour lui envoyer. Conclusion : tout cela était bien ridicule.

Et les protestations de dévouement ! — Oh ! pour le coup c’est là que la vieille rossinante à deux becs prenait le mors aux dents ! Vous répondez à cet article de ma lettre comme eût pu le faire cet écrivain du xvie siècle avant notre ère qui ayant essayé de tout, d’être un grand roi, un grand philosophe, un grand architecte, d’avoir six cents femmes, etc., en vint à s’ennuyer et à se dégoûter tellement de toutes ces choses, qu’il déclara sur ses vieux jours, toutes réflexions faites, que tout n’était que vanité.

Ce que vous me répondiez là, en style d’Ecclésiaste, je le savais bien ; je suis si bien de votre avis sur tout et même sur autre chose, que je doute fort qu’il m’arrive jamais de discuter avec vous autrement que comme Pandore avec son brigadier. Nous n’avons absolument rien à nous apprendre l’un à l’autre, pour ce qui est des choses de l’ordre moral.

— Les confidences, me dites-vous, sont inutiles.

Plus que jamais, je m’incline : j’aime à avoir des vues d’ensemble sur les personnes et les choses, j’aime à en deviner les grands traits ; quant aux détails, je les ai toujours eus en horreur.

« Affection et dévouement illimités ! » Que voulez-vous ! c’était un de ces bons mouvements, un de ces heureux éclairs à la faveur desquels on est meilleur que soi-même. Croyez bien que l’on est sincère au moment où l’on écrit ainsi. Si ce ne sont que des éclairs, à qui faut-il s’en prendre ?… Est-ce à vous et à moi, qui ne sommes aucunement responsables de la profonde imperfection de notre nature ? Est-ce à celui qui ne nous a créés que pour nous laisser à demi ébauchés, susceptibles des aspirations les plus élevées ; mais incapables d’actes qui soient en rapport avec nos conceptions ? N’est-ce à personne du tout ? Dans le doute où nous sommes à ce sujet, je crois que c’est ce qu’il y a de mieux à faire.

Merci pour ce que vous me dites de la fraîcheur de mes sentiments. Pourtant je n’en crois rien. Ils ont trop servi, ou plutôt je m’en suis trop servi, pour qu’ils ne soient pas un peu défraîchis par l’usage que j’en ai fait. Je pourrais dire que ce sont des sentiments d’occasion, et, à ce propos, je vous rappellerai que souvent on trouve de très bonnes occasions. Je vous ferai également remarquer qu’il est des choses qui gagnent en solidité ce que l’usure peut leur avoir enlevé de brillant et de fraîcheur ; comme exemple tiré du noble métier que nous exerçons tous deux, je vous citerai le vieux filin.

Il est donc bien entendu que je vous aime beaucoup. Il n’y a plus à revenir là-dessus. Une fois pour toutes, je vous déclare que vous êtes très bien doué, et qu’il serait fort malheureux que vous laissiez s’atrophier par l’acrobatie la meilleure partie de vous-même. Cela posé, je cesse de vous assommer de mon affection et de mon admiration, pour entrer dans quelques détails sur mon individu.

Je suis bien portant physiquement, et en traitement pour ce qui est du moral. — Mon traitement consiste à ne plus me tourner la cervelle à l’envers, et à mettre un régulateur à ma sensibilité. Tout est équilibre en ce monde, au-dedans de nous-même comme au dehors. Si la sensibilité prend le dessus, c’est toujours aux dépens de la raison. Plus vous serez poète, moins vous serez géomètre, et, dans la vie, il faut un peu de géométrie, et, ce qui est pis encore, beaucoup d’arithmétique. Je crois, Dieu me pardonne, que je vous écris là quelque chose qui a presque le sens commun !

Tout à vous,
PLUMKETT.


XIX
Nuit du 27 juillet, Salonique.
À neuf heures, les uns après les autres, les officiers du bord rentrent dans leurs chambres ; ils se retirent tous en me souhaitant bonne chance et bonne nuit : mon secret est devenu celui de tout le monde.

Et je regarde avec anxiété le ciel du côté du vieil Olympe, d’où partent trop souvent ces gros nuages cuivrés, indices d’orages et de pluie torrentielle.

Ce soir, de ce côté-là, tout est pur, et la montagne mythologique découpe nettement sa cime sur le ciel profond.

Je descends dans ma cabine, je m’habille et je remonte.

Alors commence l’attente anxieuse de chaque soir : une heure, deux heures se passent, les minutes se traînent et sont longues comme des nuits.

À onze heures, un léger bruit d’avirons sur la mer calme ; un point lointain s’approche en glissant comme une ombre. C’est la barque de Samuel. Les factionnaires le couchent en joue et le hèlent. Samuel ne répond rien, et cependant les fusils s’abaissent ; — les factionnaires ont une consigne secrète qui concerne lui seul, et le voilà le long du bord.

On lui remet pour moi des filets, et différents ustensiles de pêche ; les apparences sont sauvées ainsi, et je saute dans la barque, qui s’éloigne ; j’enlève le manteau qui couvrait mon costume turc et la transformation est faite. Ma veste dorée brille légèrement dans l’obscurité, la brise est molle et tiède, et Samuel rame sans bruit dans la direction de la terre.

Une petite barque est là qui stationne. — Elle contient une vieille négresse hideuse enveloppée d’un drap bleu, un vieux domestique albanais armé jusqu’aux dents, au costume pittoresque ; et puis une femme, tellement voilée qu’on ne voit plus rien d’elle-même qu’une informe masse blanche.

Samuel reçoit dans sa barque les deux premiers de ces personnages, et s’éloigne sans mot dire. Je suis resté seul avec la femme au voile, aussi muette et immobile qu’un fantôme blanc ; j’ai pris les rames, et, en sens inverse, nous nous éloignons aussi dans la direction du large. — Les yeux fixés sur elle, j’attends avec anxiété qu’elle fasse un mouvement ou un signe.

Quand, à son gré, nous sommes assez loin, elle me tend ses bras ; c’est le signal attendu pour venir m’asseoir auprès d’elle. Je tremble en la touchant, ce premier contact me pénètre d’une langueur mortelle, son voile est imprégné des parfums de l’Orient, son contact est ferme et froid.

J’ai aimé plus qu’elle une autre jeune femme que, à présent, je n’ai plus le droit de voir ; mais jamais mes sens n’ont connu pareille ivresse.



XX
La barque d’Aziyadé est remplie de tapis soyeux, de coussins et de couvertures de Turquie. On y trouve tous les raffinements de la nonchalance orientale, et il semblerait voir un lit qui flotte plutôt qu’une barque.

C’est une situation singulière que la nôtre : il nous est interdit d’échanger seulement une parole ; tous les dangers se sont donné rendez-vous autour de ce lit, qui dérive sans direction sur la mer profonde ; on dirait deux êtres qui ne se sont réunis que pour goûter ensemble les charmes enivrants de l’impossible.

Dans trois heures, il faudra partir, quand la Grande Ourse se sera renversée dans le ciel immense. Nous suivons chaque nuit son mouvement régulier, elle est l’aiguille du cadran qui compte nos heures d’ivresse.

D’ici là, c’est l’oubli complet du monde et de la vie, le même baiser commencé le soir qui dure jusqu’au matin, quelque chose de comparable à cette soif ardente des pays de sable de l’Afrique qui s’excite en buvant de l’eau fraîche et que la satiété n’apaise plus…

À une heure, un tapage inattendu dans le silence de cette nuit : des harpes et des voix de femmes ; on nous crie gare, et à peine avons-nous le temps de nous garer. Un canot de la Maria Pia passe grand train près de notre barque ; il est rempli d’officiers italiens en partie fine, ivres pour la plupart ; — il avait failli passer sur nous et nous couler.



XXI
Quand nous rejoignîmes la barque de Samuel, la Grande Ourse avait dépassé son point de plus grande inclinaison, et on entendait dans le lointain le chant du coq.

Samuel dormait, roulé dans ma couverture, à l’arrière, au fond de la barque ; la négresse dormait, accroupie à l’avant comme une macaque ; le vieil Albanais dormait entre eux deux, courbé sur ses avirons.

Les deux vieux visiteurs rejoignirent leur maîtresse, et la barque qui portait Aziyadé s’éloigna sans bruit. Longtemps je suivis des yeux la forme blanche de la jeune femme, étendue inerte à la place où je l’avais quittée, chaude de baisers, et humide de la rosée de la nuit.

Trois heures sonnaient à bord des cuirassés allemands ; une lueur blanche à l’orient profilait le contour sombre des montagnes, dont la base était perdue dans l’ombre, dans l’épaisseur de leur propre ombre, reflétée profondément dans l’eau calme. Il était impossible d’apprécier encore aucune distance dans l’obscurité projetée par ces montagnes ; seulement les étoiles pâlissaient.

La fraîcheur humide du matin commençait à tomber sur la mer ; la rosée se déposait en gouttelettes serrées sur les planches de la barque de Samuel ; j’étais vêtu à peine, les épaules seulement couvertes d’une chemise d’Albanais en mousseline légère. Je cherchais ma veste dorée ; elle était restée dans la barque d’Aziyadé. Un froid mortel glissait le long de mes bras, et pénétrait peu à peu toute ma poitrine. Une heure encore avant le moment favorable pour rentrer à bord en évitant la surveillance des hommes de garde ! J’essayai de ramer ; un sommeil irrésistible engourdissait mes bras. Alors je soulevai avec des précautions infinies la couverture qui enveloppait Samuel, pour m’étendre sans l’éveiller à côté de cet ami de hasard.

Et, sans en avoir eu conscience, en moins d’une seconde, nous nous étions endormis tous deux de ce sommeil accablant contre lequel il n’y a pas de résistance possible ; — et la barque s’en alla en dérive.

Une voix rauque et germanique nous éveilla au bout d’une heure ; la voix criait quelque chose en allemand dans le genre de ceci : « Ohé du canot ! »

Nous étions tombés sur les cuirassés allemands, et nous nous éloignâmes à force de rames ; les fusils des hommes de garde nous tenaient en joue. Il était quatre heures ; l’aube, incertaine encore, éclairait la masse blanche de Salonique, les masses noires des navires de guerre ; je rentrai à bord comme un voleur, assez heureux pour être inaperçu.



XXII
La nuit d’après (du 28 au 29), je rêvai que je quittais brusquement Salonique et Aziyadé. Nous voulions courir, Samuel et moi, dans le sentier du village turc où elle demeure, pour au moins lui dire adieu ; l’inertie des rêves arrêtait notre course ; l’heure passait et la corvette larguait ses voiles.

— Je t’enverrai de ses cheveux, disait Samuel, toute une longue natte de ses cheveux bruns.

Et nous cherchions toujours à courir.

Alors, on vint m’éveiller pour le quart ; il était minuit. Le timonier alluma une bougie dans ma chambre : je vis briller les dorures et les fleurs de soie de la tapisserie, et m’éveillai tout à fait.

Il plut par torrents cette nuit-là, et je fus trempé.



XXIII
Salonique, 29 juillet.
Je reçois ce matin à dix heures cet ordre inattendu : quitter brusquement ma corvette et Salonique : prendre passage demain sur le paquebot de Constantinople, et rejoindre le stationnaire anglais le Deerhound, qui se promène par là-bas, dans les eaux du Bosphore ou du Danube.

Une bande de matelots vient d’envahir ma chambre ; ils arrachent les tentures et confectionnent les malles.

J’habitais, tout au fond du Prince-of-Wales, un réduit blindé confinant avec la soute aux poudres. J’avais meublé d’une manière originale ce caveau, où ne pénétrait pas la lumière du soleil : sur les murailles de fer, une épaisse soie rouge à fleurs bizarres ; des faïences, des vieilleries redorées, des armes, brillant sur ce fond sombre.

J’avais passé des heures tristes, dans l’obscurité de cette chambre, ces heures inévitables du tête-à-tête avec soi-même, qui sont vouées aux remords, aux regrets déchirants du passé.



XXIV
J’avais quelques bons camarades sur le Prince-of-Wales ; j’étais un peu l’enfant gâté du bord, mais je ne tiens plus à personne, et il m’est indifférent de les quitter.

Une période encore de mon existence qui va finir, et Salonique est un coin de la terre que je ne reverrai plus.

J’ai passé pourtant des heures enivrantes sur l’eau tranquille de cette grande baie, des nuits que beaucoup d’hommes achèteraient bien cher et j’aimais presque cette jeune femme, si singulièrement délicieuse !

J’oublierai bientôt ces nuits tièdes, où la première lueur de l’aube nous trouvait étendus dans une barque, enivrés d’amour, et tout trempés de la rosée du matin.

Je regrette Samuel aussi, le pauvre Samuel, qui jouait si gratuitement sa vie pour moi, et qui va pleurer mon départ comme un enfant. C’est ainsi que je me laisse aller encore et prendre à toutes les affections ardentes, à tout ce qui y ressemble, quel qu’en soit le mobile intéressé ou ténébreux ; j’accepte, en fermant les yeux, tout ce qui peut pour une heure combler le vide effrayant de la vie, tout ce qui est une apparence d’amitié ou d’amour.



XXV
30 juillet. Dimanche.
À midi, par une journée brûlante, je quitte Salonique. Samuel vient avec sa barque, à la dernière heure, me dire adieu sur le paquebot qui m’emporte.

Il a l’air fort dégagé et satisfait. — Encore un qui m’oubliera vite !

— Au revoir, effendim, pensia poco de Samuel ! (Au revoir, monseigneur ! pense un peu à Samuel !)



XXVI
— En automne, a dit Aziyadé, Abeddin-effendi, mon maître, transportera à Stamboul son domicile et ses femmes ; si par hasard il n’y venait pas, moi seule j’y viendrais pour toi.

Va pour Stamboul, et je vais l’y attendre. Mais c’est tout à recommencer, un nouveau genre de vie, dans un nouveau pays, avec de nouveaux visages, et pour un temps que j’ignore.



XXVII
L’état-major du Prince-of-Wales exécute des effets de mouchoirs très réussis, et le pays s’éloigne, baigné dans le soleil. Longtemps on distingue la tour blanche, où, la nuit, s’embarquait Aziyadé, et cette campagne pierreuse, çà et là plantée de vieux platanes, si souvent parcourue dans l’obscurité.

Salonique n’est plus bientôt qu’une tache grise qui s’étale sur des montagnes jaunes et arides, une tache hérissée de pointes blanches qui sont des minarets, et de pointes noires qui sont des cyprès.

Et puis la tache grise disparaît, pour toujours sans doute, derrière les hautes terres du cap Kara-Bournou. Quatre grands sommets mythologiques s’élèvent au-dessus de la côte déjà lointaine de Macédoine : Olympe, Athos, Pélion et Ossa !
"""
    ,
    """
    I

M. MYRIEL
En 1815, M. Charles-François-Bienvenu Myriel était évêque de Digne. C’était un vieillard d’environ soixante-quinze ans ; il occupait le siége de Digne depuis 1806.

Quoique ce détail ne touche en aucune manière au fond même de ce que nous avons à raconter, il n’est peut-être pas inutile, ne fût-ce que pour être exact en tout, d’indiquer ici les bruits et les propos qui avaient couru sur son compte au moment où il était arrivé dans le diocèse. Vrai ou faux, ce qu’on dit des hommes tient souvent autant de place dans leur vie et souvent dans leur destinée que ce qu’ils font. M. Myriel était fils d’un conseiller au parlement d’Aix ; noblesse de robe. On contait que son père, le réservant pour hériter de sa charge, l’avait marié de fort bonne heure, à dix-huit ou vingt ans, suivant un usage assez répandu dans les familles parlementaires. Charles Myriel, nonobstant ce mariage, avait, disait-on, beaucoup fait parler de lui. Il était bien fait de sa personne, quoique d’assez petite taille, élégant, gracieux, spirituel ; toute la première partie de sa vie avait été donnée au monde et aux galanteries.

La révolution survint, les événements se précipitèrent ; les familles parlementaires, décimées, chassées, traquées, se dispersèrent. M. Charles Myriel, dès les premiers jours de la révolution, émigra en Italie. Sa femme y mourut d’une maladie de poitrine dont elle était atteinte depuis longtemps. Ils n’avaient point d’enfants. Que se passa-t-il ensuite dans la destinée de M. Myriel ? L’écroulement de l’ancienne société française, la chute de sa propre famille, les tragiques spectacles de 93, plus effrayants encore peut-être pour les émigrés qui les voyaient de loin avec le grossissement de l’épouvante, firent-ils germer en lui des idées de renoncement et de solitude ? Fut-il, au milieu d’une de ces distractions et de ces affections qui occupaient sa vie, subitement atteint d’un de ces coups mystérieux et terribles qui viennent quelquefois renverser, en le frappant au cœur, l’homme que les catastrophes publiques n’ébranleraient pas en le frappant dans son existence et dans sa fortune ? Nul n’aurait pu le dire ; tout ce qu’on savait, c’est que, lorsqu’il revint d’Italie, il était prêtre.

En 1804, M. Myriel était curé de B. (Brignolles). Il était déjà vieux, et vivait dans une retraite profonde.

Vers l’époque du couronnement, une petite affaire de sa cure, on ne sait plus trop quoi, l’amena à Paris. Entre autres personnes puissantes, il allait solliciter pour ses paroissiens M. le cardinal Fesch. Un jour que l’empereur était venu faire sa visite à son oncle, le digne curé, qui attendait dans l’antichambre, se trouva sur le passage de sa majesté. Napoléon, se voyant regarder avec une certaine curiosité par ce vieillard, se retourna, et dit brusquement :

— Quel est ce bonhomme qui me regarde ?

— Sire, dit M. Myriel, vous regardez un bonhomme, et moi je regarde un grand homme. Chacun de nous peut profiter.

L’empereur, le soir même, demanda au cardinal le nom de ce curé, et quelque temps après M. Myriel fut tout surpris d’apprendre qu’il était nommé évêque de Digne.

Qu’y avait-il de vrai, du reste, dans les récits qu’on faisait sur la première partie de la vie de M. Myriel ? Personne ne le savait. Peu de familles avaient connu la famille Myriel avant la révolution.

M. Myriel devait subir le sort de tout nouveau venu dans une petite ville où il y a beaucoup de bouches qui parlent et fort peu de têtes qui pensent. Il devait le subir, quoiqu’il fût évêque et parce qu’il était évêque. Mais, après tout, les propos auxquels on mêlait son nom n’étaient peut-être que des propos ; du bruit, des mots, des paroles, moins que des paroles, des palabres, comme dit l’énergique langue du midi.

Quoi qu’il en fût, après neuf ans d’épiscopat et de résidence à Digne, tous ces racontages, sujets de conversation qui occupent dans le premier moment les petites villes et les petites gens, étaient tombés dans un oubli profond. Personne n’eût osé en parler, personne n’eût osé s’en souvenir.

M. Myriel était arrivé à Digne accompagné d’une vieille fille, mademoiselle Baptistine, qui était sa sœur et qui avait dix ans de moins que lui.

Ils avaient pour tout domestique une servante du même âge que mademoiselle Baptistine, et appelée madame Magloire, laquelle, après avoir été la servante de M. le curé, prenait maintenant le double titre de femme de chambre de mademoiselle et femme de charge de monseigneur.

Mademoiselle Baptistine était une personne longue, pâle, mince, douce ; elle réalisait l’idéal de ce qu’exprime le mot « respectable » ; car il semble qu’il soit nécessaire qu’une femme soit mère pour être vénérable. Elle n’avait jamais été jolie ; toute sa vie, qui n’avait été qu’une suite de saintes œuvres, avait fini par mettre sur elle une sorte de blancheur et de clarté, et, en vieillissant, elle avait gagné ce qu’on pourrait appeler la beauté de la bonté. Ce qui avait été de la maigreur dans sa jeunesse était devenu, dans sa maturité, de la transparence ; et cette diaphanéité laissait voir l’ange. C’était une âme plus encore que ce n’était une vierge. Sa personne semblait faite d’ombre ; à peine assez de corps pour qu’il y eût là un sexe ; un peu de matière contenant une lueur ; de grands yeux toujours baissés ; un prétexte pour qu’une âme reste sur la terre.

Madame Magloire était une petite vieille, blanche, grasse, replète, affairée, toujours haletante, à cause de son activité d’abord, ensuite à cause d’un asthme.

À son arrivée, on installa M. Myriel en son palais épiscopal avec les honneurs voulus par les décrets impériaux qui classent l’évêque immédiatement après le maréchal de camp. Le maire et le président lui firent la première visite, et lui de son côté fit la première visite au général et au préfet.

L’installation terminée, la ville attendit son évêque à l’œuvre.

""",
"""
Au nom du Père ; et du Fi ; et du Saint-Esprit ; Ainsi soit-il.

Notre Père qui êtes aux cieux ; que votre nom soit sanctifié ; que votre règne arrive ; que votre volonté soit faite sur la terre comme au ciel. Donnez-nous aujourd’hui notre pain de chaque jour ; pardonnez-nous nos offenses comme nous pardonnons à ceux qui nous ont offensés ; ne nous laissez pas succomber à la tentation ; mais délivrez-nous du mal. Ainsi soit-il.

Je vous salue Marie, pleine de grâce ; le Seigneur est avec vous ; vous êtes bénie entre toutes les femmes ; et Jésus le fruit de vos entrailles est béni. Sainte Marie, mère de Dieu, priez pour nous pauvres pécheurs, maintenant et à l’heure de notre mort. Ainsi soit-il.

Saint Jean, mon patron ; sainte Jeanne, ma patronne ; priez pour nous ; priez pour nous.

Au nom du Père ; et du Fi ; et du Saint-Esprit ; Ainsi soit-il.

Notre père, notre père qui êtes aux cieux, de combien il s’en faut que votre nom soit sanctifié ; de combien il s’en faut que votre règne arrive.

Notre père, notre père qui êtes au royaume des cieux, de combien il s’en faut que votre règne arrive au royaume de la terre.

Notre père, notre père qui êtes au royaume des cieux, de combien il s’en faut que votre règne arrive au royaume de France.

Notre père, notre père qui êtes aux cieux, de combien il s’en faut que votre volonté soit faite ; de combien il s’en faut que nous ayons notre pain de chaque jour.

De combien il s’en faut que nous pardonnions nos offenses ; et que nous ne succombions pas à la tentation ; et que nous soyons délivrés du mal. Ainsi soit-il.

Ô mon Dieu si on voyait seulement le commencement de votre règne. Si on voyait seulement se lever le soleil de votre règne. Mais rien, jamais rien. Vous nous avez envoyé votre Fils, que vous aimiez tant, votre Fils est venu, qui a tant souffert, et il est mort, et rien, jamais rien. Si on voyait poindre seulement le jour de votre règne. Et vous avez envoyé vos saints, vous les avez appelés chacun par leur nom, vos autres fils les saints, et vos filles les saintes, et vos saints sont venus, et vos saintes sont venues, et rien, jamais rien. Des années ont passé, tant d’années que je n’en sais pas le nombre ; des siècles d’années ont passé ; quatorze siècles de chrétienté, hélas, depuis la naissance, et la mort, et la prédication. Et rien, rien, jamais rien. Et ce qui règne sur la face de la terre, rien, rien, ce n’est rien que la perdition. Quatorze siècles (furent-ils de chrétienté), quatorze siècles depuis le rachat de nos âmes. Et rien, jamais rien, le règne de la terre n’est rien que le règne de la perdition, le royaume de la terre n’est rien que le royaume de la perdition. Vous nous avez envoyé votre Fils et les autres saints. Et rien ne coule sur la face de la terre, qu’un flot d’ingratitude et de perdition. Mon Dieu, mon Dieu, faudra-t-il que votre Fils soit mort en vain. Il serait venu ; et cela ne servirait de rien. C’est pire que jamais. Seulement si on voyait seulement se lever le soleil de votre justice. Mais on dirait, mon Dieu, mon Dieu, pardonnez-moi, on dirait que votre règne s’en va. Jamais on n’a tant blasphémé votre nom. Jamais on n’a tant méprisé votre volonté. Jamais on n’a tant désobéi. Jamais notre pain ne nous a tant manqué ; et s’il ne manquait qu’à nous, mon Dieu, s’il ne manquait qu’à nous ; et s’il n’y avait que le pain du corps qui nous manquait, le pain de maïs, le pain de seigle et de blé ; mais un autre pain nous manque ; le pain de la nourriture de nos âmes ; et nous sommes affamés d’une autre faim ; de la seule faim qui laisse dans le ventre un creux impérissable. Un autre pain nous manque. Et au lieu que ce soit le règne de votre charité, le seul règne qui règne sur la face de la terre, de votre terre, de la terre votre création, au lieu que ce soit le règne du royaume de votre charité, le seul règne qui règne, c’est le règne du royaume impérissable, du péché. Encore si l’on voyait le commencement de vos saints, si l’on voyait poindre le commencement du règne de vos saints. Mais qu’est-ce qu’on a fait, mon Dieu, qu’est-ce qu’on a fait de votre créature, qu’est-ce qu’on a fait de votre création ? Jamais il n’a été fait tant d’offenses ; et jamais tant d’offenses ne sont mortes impardonnées. Jamais le chrétien n’a fait tant d’offense au chrétien, et jamais à vous, mon Dieu, jamais l’homme ne vous a fait tant d’offense. Et jamais tant d’offense n’est morte impardonnée. Sera-t-il dit que vous nous aurez envoyé en vain votre Fils, et que votre Fils aura souffert en vain, et qu’il sera mort. Et faudra-t-il que ce soit en vain qu’il se sacrifie et que nous le sacrifions tous les jours. Sera-ce en vain qu’une croix a été dressée un jour et que nous autres nous la redressons tous les jours. Qu’est-ce qu’on a fait du peuple chrétien, mon Dieu, de votre peuple. Et ce ne sont plus seulement les tentations qui nous assiègent, mais ce sont les tentations qui triomphent ; et ce sont les tentations qui règnent ; et c’est le règne de la tentation ; et le règne des royaumes de la terre est tombé tout entier au règne du royaume de la tentation ; et les mauvais succombent à la tentation du mal, de faire du mal ; de faire du mal aux autres ; et pardonnez-moi, mon Dieu, de vous faire du mal à vous ; mais les bons, ceux qui étaient bons, succombent à une tentation infiniment pire : à la tentation de croire qu’ils sont abandonnés de vous. Au nom du Père, et du Fils, et du Saint-Esprit, mon Dieu délivrez-nous du mal, délivrez-nous du mal. S’il n’y a pas eu encore assez de saintes et assez de saints, envoyez-nous en d’autres, envoyez-nous en autant qu’il en faudra ; envoyez-nous en tant que l’ennemi se lasse. Nous les suivrons, mon Dieu. Nous ferons tout ce que vous voudrez. Nous ferons tout ce qu’ils voudront. Nous ferons tout ce qu’ils nous diront de votre part. Nous sommes vos fidèles, envoyez-nous vos saints ; nous sommes vos brebis, envoyez-nous vos bergers ; nous sommes le troupeau, envoyez-nous les pasteurs. Nous sommes des bons chrétiens, vous savez que nous sommes des bons chrétiens. Alors comment que ça se fait que tant de bons chrétiens ne fassent pas une bonne chrétienté. Il faut qu’il y ait quelque chose qui ne marche pas. Si vous nous envoyiez, si seulement vous vouliez nous envoyer l’une de vos saintes. Il y en a bien encore. On dit qu’il y en a. On en voit. On en sait. On en connaît. Mais on ne sait pas comment que ça se fait. Il y a des saintes, il y a de la sainteté, et ça ne marche pas tout de même. Il y a quelque chose qui ne marche pas. Il y a des saintes, il y a de la sainteté et jamais le règne du royaume de la perdition n’avait autant dominé sur la face de la terre. Il faudrait peut-être autre chose, mon Dieu, vous savez tout. Vous savez ce qui nous manque. Il nous faudrait peut-être quelque chose de nouveau, quelque chose qu’on n’aurait encore jamais vu. Quelque chose qu’on n’aurait encore jamais fait. Mais qui oserait dire, mon Dieu, qu’il puisse encore y avoir du nouveau après quatorze siècles de chrétienté, après tant de saintes et tant de saints, après tous vos martyrs, après la passion et la mort de votre Fils.

Elle se rassied et recommence à filer.
Enfin ce qu’il nous faudrait, mon Dieu, il faudrait nous envoyer une sainte… qui réussisse.

Une voix monte de la vallée, vient, s’approche. C’est Hauviette qui vient. Elle monte du bourg par le sentier. Elle chante :
Les Anglais n’auront pas
La tour de Saint-Nique Nique
Les Anglais n’auront pas
La Tour de Saint-Nicolas.

Jeannette
Mon Dieu, mon Dieu, nous serons bien sages, nous serons bien soumis, nous serons bien obéissants. Nous serons bien fidèles.

Mon Dieu, mon Dieu, nous sommes vos enfants, nous sommes vos enfants.

Hauviette apparaît venant.
Jeannette
Mon Dieu, mon Dieu, qu’est-ce qu’on a fait de votre peuple.

Entre Hauviette. Elle commence toute chantante, comme si ses paroles ne fussent que la suite naturelle de sa chanson, et ne redescend que par degrés à son propos ordinaire.
Hauviette
— Bonjour, Jeannette.

Jeannette
— Bonjour, Hauviette.

Un silence.
Hauviette
— Tu faisais ta prière ?

Jeannette
Un assez long silence.
— Je faisais ma prière. Il y a tant de manque. Il y a tant à demander.

Hauviette
— Le bon Dieu sait bien ce qu’il nous faut, le bon Dieu sait bien ce qui nous manque.

Puis toujours comme bavardant :
Tu faisais ta prière. Ne t’en excuse pas. Ne t’en défends pas. Je ne te le reproche pas. Tu n’as pas besoin de t’en défendre. Il n’y a pas de mal à ça. Tu n’as pas besoin d’avoir honte.

Jeannette
Un silence.
— Je faisais ma prière. Toi aussi, Hauviette, tu fais ta prière.

Hauviette
— Moi je suis bonne chrétienne comme tout le monde, je fais ma prière comme tout le monde, je suis bonne paroissienne comme tout le monde. Hein, je fais ma prière tous les matins et tous les soirs, mon Notre Père et mon Je vous salue Marie, pour commencer et pour finir ma journée. Et puis ça m’emplit ma journée ; hein ça suffit pour m’emplir toute ma journée, pour me faire tenir ma journée ; ça me tient le cœur toute la journée. Ça me fait passer toute ma journée. Je suis une bonne chrétienne. On fait ses deux prières comme on fait ses trois repas. C’est aussi naturel. C’est la même chose. C’est ça qui fait la journée. On ne mange pas toute la journée. On ne fait pas sa prière toute la journée. Je suis une bonne paroissienne. Je fais aussi ma prière à l’Angelus du matin et à l’Angelus du soir, quand même que je ferais n’importe quoi, je m’arrête de le faire, naturellement, pour répondre à la cloche. Je suis une bonne paroissienne de la paroisse de Domremy. Je vais au catéchisme comme tout le monde. Et le dimanche je vais au bourg à la messe et à l’église comme tout le monde. Seulement, voilà, moi, il ne faut pas que le dimanche ressemble aux jours de la semaine et que les jours de la semaine ressemblent au dimanche. Et que les heures de la prière ressemblent aux autres heures de la journée et les autres heures de la journée aux heures de la prière. Sans ça, alors, autrement, c’est comme si il (n’)y avait pas de dimanche. Dans la semaine. Et pas d’heures de la prière. Dans la journée. Alors c’est pas la peine d’avoir un dimanche. Il ne faut pas travailler le dimanche. Mais alors il faut travailler (dans) la semaine. Il y a un jour pour le bon Dieu, et les autres jours c’est pour travailler. Travailler, c’est prier. Je vais au catéchisme le dimanche matin avant la messe. Il y a temps pour tout. À chaque heure suffit sa peine. Et son travail. Chaque chose en son temps. Travailler, prier, c’est tout naturel, ça, ça se fait tout seul.

Il faut que le dimanche ressorte dans la semaine et que l’Angelus et l’heure de la prière sorte dans la journée.

Oui Jeannette, ma belle, je fais ma prière, mais toi tu ne sors pas de la faire, tu la fais tout le temps, tu n’en sors pas, tu la fais à toutes les croix du chemin, l’église ne te suffit pas. Jamais les croix des chemins n’avaient tant servi…

""",
]
    

In [36]:
import requests
import json

LLAMA_URL = "http://sanroque:8989/v1/chat/completions"
MODEL_NAME = "phi4"  # adapter si besoin

def phi4_extract_expressions(texte: str):
    """
    Envoie le texte au serveur llama.cpp pour extraire
    un tableau JSON d'expressions clés.
    """
    system_prompt = """
Vous êtes un indexeur professionnel. Votre tâche est d'extraire les expressions caractérisant les textes qui vous sont soumis.
Les règles suivantes doivent être respectées ABSOLUMENT :
les expressions doivent :
SE COMPOSER AU MINIMUM DE DEUX MOTS, 
elles ne doivent pas :
SE COMPOSER DE MOTS ISOLÉS OU SEULEMENT PRÉCÉDÉ D'UN DÉTERMINANT, par exemple les expressions : "un soir" ou "soir" sont interdites,
CONTENIR DE VERBES CONJUGUÉS,
COMMENCER PAR UN ARTICLE (le, la, les, un, une, des) ou un adjectif démonstratif ou possessif ce, son, sa, etc.).
Répondez UNIQUEMENT par un tableau JSON de chaînes, sans commentaire.
Incluez UNIQUEMENT les expressions qui apparaissent à l'identique dans le texte.
RESPECTEZ CES RÈGLES SCRUPULEUSEMNT;        """
    

    user_prompt = (
        "Quelles sont les expressions clés contenues à l'identique dans ce texte :\n"
        f"{texte}"
    )

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    #print(f"{system_prompt}\n{max_tokens}")
    resp = requests.post(LLAMA_URL, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()

    raw_output = data["choices"][0]["message"]["content"].strip()

    # parse du JSON renvoyé par le modèle
    return json.loads(raw_output)


"""
# Exemple d’utilisation
if __name__ == "__main__":
    texte = (
        "III\nLes gouvernements de France et d’Allemagne avaient exigé ces exécutions "
        "d’ensemble, comme réparation de ce massacre des consuls qui fit du bruit en "
        "Europe au début de la crise orientale. Toutes les nations européennes avaient "
        "envoyé sur rade de Salonique d’imposants cuirassés. L’Angleterre s’y était une "
        "des premières fait représenter, et c’est ainsi que j’y étais venu moi-même, sur "
        "l’une des corvettes de Sa Majesté."
    )

    expressions = phi4_extract_expressions(texte)
    print(expressions)
"""
print(phi4_extract_expressions(textes[0]))


['têtes entassées en ces charniers', 'chambres aux deniers', 'porte-paniers', 'évêques ou de lanterniers', 'seigneuries leur sont ravies', 'clerc ni maître']


In [37]:
import unicodedata
from typing import List, Union

def _normaliser(s: str) -> str:
    """
    Normalisation simple :
    - minuscules
    - suppression des accents
    - réduction des espaces
    """
    s = s.lower()
    s = "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )
    return " ".join(s.split())


def filtrer_expressions(texte: Union[str, List[str]], expressions: List[str]) -> List[str]:
    """
    Retourne une liste dédoublonnée d'expressions qui apparaissent
    (après normalisation) dans le texte.
    """

    # Si on vous passe une liste de morceaux de texte, on les recolle
    if isinstance(texte, list):
        texte = " ".join(texte)

    if not isinstance(texte, str):
        raise TypeError(f"texte doit être str ou list[str], reçu {type(texte)}")

    texte_norm = _normaliser(texte)

    vues = set()
    valides: List[str] = []

    for expr in expressions:
        if not isinstance(expr, str):
            continue

        expr_clean = expr.strip()
        if not expr_clean:
            continue

        # dédoublonnage sur la forme "brute"
        if expr_clean in vues:
            continue
        vues.add(expr_clean)

        expr_norm = _normaliser(expr_clean)

        # on vérifie la présence de la chaîne normalisée telle quelle
        if expr_norm in texte_norm:
            valides.append(expr_clean)

    return valides


In [38]:
import time
stime = time.time()
vides = 0
erreurs = 0
npassages = 0

for passage in chunk(textes[1]):
    npassages += 1
    expressions = []
    try:
        expressions = phi4_extract_expressions(passage)
    except Exception as e:
        print(f"Erreur lors de l'extraction des expressions : {e}")
        continue
    
    expressions_filtrées = filtrer_expressions(passage,expressions)
    if len(expressions_filtrées) == 0:
        vides += 1
    elif len(expressions_filtrées) != len(expressions):
        erreurs += 1

    print(passage)
    print("____________________________________________________")
    print(expressions)
    print(expressions_filtrées)
    print(f"{vides} sans résultat et {erreurs} avec des erreurs {npassages} passages")
    print(time.time()-stime)
    print("\n==================================================\n")

print(f"Temps d'exécution {time.time()-stime}")
print(f"{vides/npassages} vides {erreurs/npassages} erreurs {npassages} passages")

… Une belle journée de mai, un beau soleil, un ciel pur… Quand les canots étrangers arrivèrent, les bourreaux, sur les quais, mettaient la dernière main à leur œuvre : six pendus exécutaient en présence de la foule l’horrible contorsion finale… Les fenêtres, les toits étaient encombrés de spectateurs ; sur un balcon voisin, les autorités turques souriaient à ce spectacle familier.
____________________________________________________
['une belle journée de mai', 'un beau soleil', 'un ciel pur', 'les canots étrangers', 'les bourreaux', 'les quais', 'six pendus', 'la foule', 'l’horrible contorsion finale', 'les fenêtres', 'les toits', 'les spectateurs', 'un balcon voisin', 'les autorités turques']
['une belle journée de mai', 'un beau soleil', 'un ciel pur', 'les canots étrangers', 'les bourreaux', 'les quais', 'six pendus', 'la foule', 'l’horrible contorsion finale', 'les fenêtres', 'les toits', 'un balcon voisin', 'les autorités turques']
0 sans résultat et 1 avec des erreurs 1 passages

KeyboardInterrupt: 

In [ ]:
import time
stime = time.time()
vides = 0
erreurs = 0
npassages = 0

for passage in chunk(textes[0]):
    npassages += 1
    expressions = phi4_extract_expressions(passage)
    expressions_filtrées = filtrer_expressions(passage,expressions)
    if len(expressions_filtrées) == 0:
        vides += 1
    elif len(expressions_filtrées) != len(expressions):
        erreurs += 1

    print(passage)
    print("____________________________________________________")
    print(expressions)
    print(expressions_filtrées)
    print(f"{vides} sans résultat et {erreurs} avec des erreurs")
    print(time.time()-stime)
    print("\n==================================================\n")

print(f"Temps d'exécution {time.time()-stime}")
print(f"{vides/npassages} vides {erreurs/npassages} erreurs {npassages} passages")

In [ ]:
texte1 = "Un attroupement fut aussitôt formé autour de moi ; c’était une bande de ces hommes qui vivent à la belle étoile sur les quais de Salonique, bateliers ou portefaix, qui désiraient savoir pourquoi j’étais resté à terre et attendaient là, dans l’espoir que peut-être j’aurais besoin de leurs services."
expr1 = ['camps de concentration', 'histoire de la Shoah', 'résistance', 'traîtres', 'témoignages']
texte2 = """
Le gouvernement du sultan avait fait peu de frais pour l’appareil du supplice ; les potences étaient si basses que les pieds nus des condamnés touchaient la terre. Leurs ongles crispés grinçaient sur le sable. II
L’exécution terminée, les soldats se retirèrent et les morts restèrent jusqu’à la tombée du jour exposés aux yeux du peuple. Les six cadavres, debout sur leurs pieds, firent, jusqu’au soir, la hideuse grimace de la mort au beau soleil de Turquie, au milieu de promeneurs indifférents et de groupes silencieux de jeunes femmes.
"""
expr3 = ['appareil du supplice', 'potences basses', 'exécution terminée', 'soldats se retirèrent', 'cadavres', 'promeneurs indifférents']
14.3519868850708
texte3 = """
Je me croyais si parfaitement seul, que j’éprouvai une étrange impression en apercevant près de moi, derrière d’épais barreaux de fer, le haut d’une tête humaine, deux grands yeux verts fixés sur les miens. Les sourcils étaient bruns, légèrement froncés, rapprochés jusqu’à se rejoindre ; l’expression de ce regard était un mélange d’énergie et de naïveté ; on eût dit un regard d’enfant, tant il avait de fraîcheur et de jeunesse.
"""
expr3= ['reconnaissance', 'reconnaissance', 'reconnaissance']

In [ ]:
filtrer_expressions(texte1,expr1)


In [ ]:
filtrer_expressions(texte2,expr2)

In [ ]:
filtrer_expressions(texte3,expr3)


In [ ]:
import json
from typing import Iterator, Dict, List

def iter_chunks_from_jsonl(path: str) -> Iterator[Dict]:
    """
    Itère sur un fichier JSONL et applique `chunk` sur le champ 'contenu'.
    Yields un dictionnaire enrichi avec la clé 'chunks'.
    """
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Erreur JSON ligne {lineno}") from e

            contenu = record.get("contenu", "")
            if not isinstance(contenu, str):
                raise TypeError(f"'contenu' n'est pas une chaîne ligne {lineno}")

            record["chunks"] = chunk(contenu)
            yield record


In [ ]:
import json
import time

def chunker_jsonl(
    input_path: str,
    output_path: str,
):
    """
    Lit un fichier JSONL, applique `chunk` sur le champ 'contenu',
    et écrit un nouveau JSONL avec une clé 'chunks'.
    """

    doc = 0
    stime = time.time()
    
    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for lineno, line in enumerate(fin, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
                doc += 1
            except json.JSONDecodeError as e:
                raise ValueError(f"Erreur JSON ligne {lineno}") from e

            contenu = record.get("contenu", "")
            if not isinstance(contenu, str):
                raise TypeError(f"'contenu' n'est pas une chaîne ligne {lineno}")

            out = {
                "source": record.get("source"),
                "titre": record.get("titre"),
                "chunks": chunk(contenu),
            }

            fout.write(json.dumps(out, ensure_ascii=False))
            fout.write("\n")
            
            if doc %10 == 0:
                print(f"{doc} {time.time()-stime}")


In [ ]:
chunker_jsonl(
    input_path="corpus_wikipedia.jsonl",
    output_path="corpus_wikipedia_chunked.jsonl",
)